In [10]:
import os
import re
import subprocess
import pandas as pd

# Clone the repository if not already cloned
repo_url = "https://github.com/OpenZeppelin/openzeppelin-contracts.git"
repo_path = "openzeppelin-contracts"  # Adjust the path if needed

if not os.path.exists(repo_path):
    subprocess.run(["git", "clone", repo_url, repo_path])

# Function to extract require statements from a file
def extract_require_statements(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    
    # Regex to find require statements
    require_pattern = re.compile(r'require\s*\((.*?)\)\s*;')
    return require_pattern.findall(content)

# Traverse the directory and extract require statements
requires = []
for root, dirs, files in os.walk(repo_path):
    for file in files:
        if file.endswith(".sol"):
            file_path = os.path.join(root, file)
            requires.extend(extract_require_statements(file_path))

# Convert the results to a DataFrame and save to a CSV file
df = pd.DataFrame(requires, columns=["Require Statement"])
df.to_csv("extracted_require_statements.csv", index=False)

# Optionally, print the first few entries
print(df.head(10))


                                 Require Statement
0        false, "InitializableMock forced failure"
1            false, "DummyImplementation reverted"
2                              msg.sender == token
3         success, "ReentrancyAttack: failed call"
4                        recoveredSigner == signer
5                   abi.decode(results[i], (bool))
6           success, "ReentrancyMock: failed call"
7                        _reentrancyGuardEntered()
8                       !_reentrancyGuardEntered()
9  success, "ReentrancyTransientMock: failed call"


In [ ]:
import os
import re
import subprocess
import pandas as pd

# List of repositories to analyze
repositories = [
    {"url": "https://github.com/OpenZeppelin/openzeppelin-contracts", "path": "openzeppelin-contracts"},
    {"url": "https://github.com/transmissions11/solmate", "path": "solmate"},
    {"url": "https://github.com/dapphub/ds-token", "path": "ds-token"},
    {"url": "https://github.com/curvefi/curve-contract", "path": "curve-contract"},
    {"url": "https://github.com/Uniswap/uniswap-v2-core", "path": "uniswap-v2-core"},
    {"url": "https://github.com/Uniswap/uniswap-v3-core", "path": "uniswap-v3-core"},
    {"url": "https://github.com/aave/protocol-v2", "path": "aave-protocol-v2"},
    {"url": "https://github.com/aave/protocol-v3", "path": "aave-protocol-v3"},
    {"url": "https://github.com/balancer-labs/balancer-core", "path": "balancer-core"},
    {"url": "https://github.com/Synthetixio/synthetix", "path": "synthetix"},
    {"url": "https://github.com/compound-finance/compound-protocol", "path": "compound-protocol"},
    {"url": "https://github.com/smartcontractkit/chainlink", "path": "chainlink"},
]

# Function to clone a repository
def clone_repository(repo_url, repo_path):
    if not os.path.exists(repo_path):
        subprocess.run(["git", "clone", repo_url, repo_path])

# Function to extract require statements from a file
def extract_require_statements(file_path):
    with open(file_path, 'r') as file:
        content = file.read()
    
    # Regex to find require statements
    require_pattern = re.compile(r'require\s*\((.*?)\)\s*;')
    requires = require_pattern.findall(content)
    
    # Regex to identify calls to view functions
    view_function_pattern = re.compile(r'(\w+)\s*\(.*?\)')
    view_function_calls = []
    
    for req in requires:
        matches = view_function_pattern.findall(req)
        for match in matches:
            if match not in ["msg", "block", "tx"]:  # Exclude global objects
                view_function_calls.append(req)
                break
    
    return view_function_calls

# Function to traverse a directory and extract require statements
def analyze_repository(repo_url, repo_path):
    clone_repository(repo_url, repo_path)
    requires = []
    for root, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith(".sol"):
                file_path = os.path.join(root, file)
                requires.extend(extract_require_statements(file_path))
    
    # Convert the results to a DataFrame and save to a CSV file
    df = pd.DataFrame(requires, columns=["Require Statement"])
    output_path = os.path.join(repo_path, "extracted_require_statements.csv")
    df.to_csv(output_path, index=False)
    print(f"Extracted require statements saved to {output_path}")
    print(df.head(10))

# Analyze all repositories
for repo in repositories:
    analyze_repository(repo["url"], repo["path"])


In [ ]:
df